# Human activity recognition with a Gaussian HMM

I have prepared the implementation and checked it with synthetic fixtures. **I have not collected personal sensor data or videos yet. This notebook contains no real activity performance claim.** The individual formative is due 1 November 2026 at 23:59 CAT.

The implementation is in `gaussian_hmm.py`, `features.py`, and `run_activity.py`, alongside this notebook. The README describes the canonical recording format and the current gaps in the submission.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import unittest
import numpy as np
from gaussian_hmm import GaussianHMM
from features import ACTIVITIES, FEATURES, fit_scaler
from run_activity import run
print("Activities:", ACTIVITIES)
print("Features:", FEATURES)


Activities: ('standing', 'walking', 'jumping', 'still')
Features: ('acc_mean', 'acc_std', 'acc_rms', 'gyro_mean', 'gyro_std', 'gyro_rms', 'acc_sma', 'acc_peak_hz', 'acc_spectral_energy', 'gyro_peak_hz', 'gyro_spectral_energy')


## Why these algorithms fit the task

The hidden state represents one activity. Each observation is an 11-dimensional vector from a two-second sensor window. The emission model is a diagonal multivariate Gaussian. Labelled training recordings initialize activity-specific means and variances; Baum–Welch subsequently estimates the latent-state posterior and updates the model.

The forward recursion sums over all paths to obtain sequence likelihood. The backward recursion supplies the future evidence needed for posterior responsibilities. Baum–Welch uses $\gamma_t(i)=P(Z_t=i\mid X)$ and $\xi_t(i,j)=P(Z_t=i,Z_{t+1}=j\mid X)$ for expected state occupancies and transitions. Each file contributes independent sufficient statistics: no transition is inserted at a file boundary. Log-domain sums prevent underflow, a variance floor prevents singular densities, and relative likelihood improvement determines convergence.

Viterbi maximizes the probability density of a complete state path. It provides the predicted activities used in the confusion matrix, sensitivity, specificity, and accuracy. After training, state identities are aligned using training labels only. No unseen labels participate in fitting, scaling, or state alignment.

In [2]:
suite = unittest.defaultTestLoader.discover("tests", pattern="test_activity.py")
verification = unittest.TextTestRunner(verbosity=2).run(suite)
assert verification.wasSuccessful(), "Implementation checks failed"
print("Synthetic implementation checks passed; no real activity dataset was evaluated.")


test_bad_timestamps_and_duplicate_test_data_are_rejected (test_activity.ActivityTests.test_bad_timestamps_and_duplicate_test_data_are_rejected) ... 

ok


test_baum_welch_likelihood_and_recording_boundaries (test_activity.ActivityTests.test_baum_welch_likelihood_and_recording_boundaries) ... 

ok


test_complete_pipeline_writes_measured_outputs_for_unseen_files (test_activity.ActivityTests.test_complete_pipeline_writes_measured_outputs_for_unseen_files) ... 

...

ok


test_single_sample_sequences_keep_transition_rows (test_activity.ActivityTests.test_single_sample_sequences_keep_transition_rows) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 1.307s

OK


Synthetic implementation checks passed; no real activity dataset was evaluated.


## Data preparation and feature choices

I keep recordings separate and reserve a genuinely new session for evaluation. The input is a synchronized CSV with `timestamp,ax,ay,az,gx,gy,gz`; numeric timestamp units must be declared. Sampling rate, duration, jitter, and duplicate content are checked for every file. A low-pass filter precedes downsampling to a common rate. The default is 20 Hz; it must be checked against the actual phone recordings.

Two-second windows resolve FFT frequencies in 0.5 Hz steps. A one-second hop provides temporal resolution without mixing training and test recordings. The seven time-domain features describe acceleration/rotation magnitude and variability, including acceleration SMA. Four FFT features describe non-DC rhythm and integrated spectral energy for acceleration and gyroscope magnitudes. A Hann window reduces spectral leakage. Z-score normalization is fitted on training windows only. The README gives the feature-specific rationale.

The Gaussian independence assumption is approximate because several features are correlated. Standing with a held phone and a phone resting on a surface may also differ due to placement rather than activity. I must inspect this confound using real data.

In [3]:
DATA_ROOT = Path("recordings")
if not (DATA_ROOT / "train").exists() or not (DATA_ROOT / "test").exists():
    print("Real recordings not available: collect the four activities and reserve unseen files.")
    print("Expected: recordings/{train,test}/{standing,walking,jumping,still}/*.csv")
    print("No results, metrics, or personal reflection have been generated.")
else:
    output = Path("results") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    evaluation = run(DATA_ROOT, output, hz=20, time_unit="s")
    print("Measured evaluation saved to:", output)
    print("Overall window accuracy:", evaluation["overall_accuracy"])
    for activity in evaluation["per_activity"]:
        print(activity)


Real recordings not available: collect the four activities and reserve unseen files.
Expected: recordings/{train,test}/{standing,walking,jumping,still}/*.csv
No results, metrics, or personal reflection have been generated.


## Interpretation to complete after collection

The pipeline writes recording examples, transition probabilities, marginal emission densities, convergence, an unseen-file confusion matrix, and decoded state sequences. The metrics distinguish overall accuracy from one-vs-rest accuracy and leave sensitivity undefined for activities absent from the unseen set. Overlapping windows are correlated; their count is not a count of independent recordings or participants.

Pure-activity recordings demonstrate state persistence but do not measure switches between different activities. I must not interpret off-diagonal transition estimates as observed real-world behavior without collecting a session with actual transitions and their timestamps.

The 4–5 page report and its personal reflection must use the actual experiment: which activities were confused, how rates/noise/placement affected performance, and what additional data would improve the model. Four whole-body recording videos are required, one for each activity. The brief's sample count conflicts with the excellent rubric's 50 files per activity; that must be clarified before collection is considered complete. No report with invented measurements or video links is included.